In [1]:
#SVM (Baseline Model)
# Install required libraries
!pip install scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print(" Libraries loaded!")


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 Libraries loaded!


In [2]:
# Load train, val, test splits
train_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/train_final.csv')
val_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/val_final.csv')
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')

# Combine headline + content as input text
train_df['text'] = train_df['headline'] + ' ' + train_df['content']
val_df['text'] = val_df['headline'] + ' ' + val_df['content']
test_df['text'] = test_df['headline'] + ' ' + test_df['content']

print("=== Data Loaded ===")
print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

print("\n=== Train Labels ===")
print(train_df['label'].value_counts())

=== Data Loaded ===
Train: 10500
Val:   2250
Test:  2250

=== Train Labels ===
label
fake         3500
authentic    3500
ai_fake      3500
Name: count, dtype: int64


In [4]:
# Remove NaN values from text column
train_df['text'] = train_df['text'].fillna('').astype(str)
val_df['text'] = val_df['text'].fillna('').astype(str)
test_df['text'] = test_df['text'].fillna('').astype(str)

# Remove empty rows
train_df = train_df[train_df['text'].str.strip() != '']
val_df = val_df[val_df['text'].str.strip() != '']
test_df = test_df[test_df['text'].str.strip() != '']

print(" NaN removed!")
print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

 NaN removed!
Train: 10499
Val:   2250
Test:  2250


In [5]:
# Build SVM pipeline with TF-IDF
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        max_features=50000,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(max_iter=2000))
])

# Train on training data
print("Training SVM...")
svm_pipeline.fit(train_df['text'], train_df['label'])
print(" Training complete!")

# Validate on val set
val_pred = svm_pipeline.predict(val_df['text'])
val_f1 = f1_score(val_df['label'], val_pred, average='macro')
print(f"\nValidation Macro F1: {val_f1*100:.2f}%")

Training SVM...
 Training complete!

Validation Macro F1: 91.85%


In [6]:
# Final evaluation on test set
test_pred = svm_pipeline.predict(test_df['text'])

print("=== SVM Test Results ===")
print(classification_report(
    test_df['label'],
    test_pred,
    target_names=['authentic', 'fake', 'ai_fake']
))

test_f1 = f1_score(test_df['label'], test_pred, average='macro')
print(f"Test Macro F1: {test_f1*100:.2f}%")

=== SVM Test Results ===
              precision    recall  f1-score   support

   authentic       0.98      0.98      0.98       750
        fake       0.91      0.88      0.89       750
     ai_fake       0.87      0.90      0.88       750

    accuracy                           0.92      2250
   macro avg       0.92      0.92      0.92      2250
weighted avg       0.92      0.92      0.92      2250

Test Macro F1: 92.00%


In [8]:
import pickle

# Save SVM model
with open('C:/Users/Riyad/projects/fake_news/svm_model.pkl', 'wb') as f:
    pickle.dump(svm_pipeline, f)

print(" SVM model saved!")
print("\n=== Results Summary ===")
print(f"Model:    TF-IDF + LinearSVC")
print(f"Macro F1: 92.00%")

 SVM model saved!

=== Results Summary ===
Model:    TF-IDF + LinearSVC
Macro F1: 92.00%
